In [4]:
import immrax as irx
import jax
import jax.numpy as jnp

class Sys (irx.System) :
    def __init__ (self) :
        self.evolution = 'continuous'
        self.xlen = 3
    def f (self, t, x, u) :
        x1, x2, x3 = x.ravel()
        return jnp.array([
            -x1 + x3,
            x1**2 - x2 - 2*x1*x3 + x3,
            -x2 + u[0]
        ])

sys = Sys()

In [5]:
jacM = irx.jacM(sys.f)
mjacM = irx.mjacM(sys.f)

it = irx.interval(0.)
ix = irx.icentpert(jnp.zeros(3), 0.25)
iu = irx.icentpert(jnp.zeros(1), 0.)
mMt, mMx, mMu = mjacM(it, ix, iu, permutations=irx.Permutation((0,1,2,3,4)))[0]
jMt, jMx, jMu = jacM(it, ix, iu)

print(f'{mMx=}')
print(f'{mMu=}')
print(f'{jMx=}')

K = jnp.array([[-1., -1., -1.]])
Mx_c = irx.get_sparse_corners(mMx)
# Mx_c = irx.get_sparse_corners(jMx)
print(len(Mx_c))

mMx=[[-1.   0.   1. ]
 [-0.5 -1.   0.5]
 [ 0.  -1.   0. ]] <= x <= [[-1.   0.   1. ]
 [ 0.5 -1.   1.5]
 [ 0.  -1.   0. ]]
mMu=[[0.]
 [0.]
 [1.]] <= x <= [[0.]
 [0.]
 [1.]]
jMx=[[-1.   0.   1. ]
 [-1.  -1.   0.5]
 [ 0.  -1.   0. ]] <= x <= [[-1.   0.   1. ]
 [ 1.  -1.   1.5]
 [ 0.  -1.   0. ]]
4


In [6]:
import cvxpy as cp
import numpy as np

print(Mx_c)
Mu = mMu.lower

P = cp.Variable((3,3), symmetric=True)
# P = cp.Parameter((3,3), symmetric=True)
# P.value = np.array([[1., 0., 0.], [0., 1., 0.], [0., 0., 1.]])
# c = cp.Variable(1)
c = -0.5
eps = 0.01
cons = [P >> eps]
cons.extend([(Mx_c[i] + Mu@K).T @ P + P @ (Mx_c[i] + Mu@K) << 2*c*P for i in range(len(Mx_c))])

prob = cp.Problem(cp.Minimize(c), cons)
# prob.solve(verbose=True)
prob.solve()
print(prob.status)
print(P.value)
P = P.value
# print(c.value)

for i in range(len(Mx_c)):
    print(Mx_c[i].T @ P + P @ Mx_c[i])

eigs = jnp.real(jnp.linalg.eigvals(P))
print(eigs)
print(jnp.linalg.eig(P))

# print(jnp.linalg.eigvals(Jtest@P + P@Jtest.T))


[Array([[-1. ,  0. ,  1. ],
       [-0.5, -1. ,  0.5],
       [ 0. , -1. ,  0. ]], dtype=float32), Array([[-1. ,  0. ,  1. ],
       [-0.5, -1. ,  1.5],
       [ 0. , -1. ,  0. ]], dtype=float32), Array([[-1. ,  0. ,  1. ],
       [ 0.5, -1. ,  0.5],
       [ 0. , -1. ,  0. ]], dtype=float32), Array([[-1. ,  0. ,  1. ],
       [ 0.5, -1. ,  1.5],
       [ 0. , -1. ,  0. ]], dtype=float32)]
optimal
[[ 0.024655    0.00068271 -0.00276363]
 [ 0.00068271  0.05545487  0.00519891]
 [-0.00276363  0.00519891  0.02863157]]
[[-0.04999271 -0.02632922  0.02516053]
 [-0.02632922 -0.12130755 -0.00542034]
 [ 0.02516053 -0.00542034 -0.00032835]]
[[-0.04999271 -0.02632922  0.02584324]
 [-0.02632922 -0.12130755  0.05003453]
 [ 0.02584324  0.05003453  0.01006947]]
[[-0.04862728  0.02912564  0.03035944]
 [ 0.02912564 -0.12130755 -0.00542034]
 [ 0.03035944 -0.00542034 -0.00032835]]
[[-0.04862728  0.02912564  0.03104215]
 [ 0.02912564 -0.12130755  0.05003453]
 [ 0.03104215  0.05003453  0.01006947]]
[0.022941